# Stratospheric wind: exploratory analysis

What this notebook is for: understanding the 50 hPa wind field over South Asia
well enough to justify the forecasting setup that follows. Specifically, whether
the series is predictable at all at 6-24 h, what the seasonal structure is, and
whether altitude gives an airship a second control lever.

Everything heavy lives in `src/`. This notebook is the narrative; run
`python src/eda.py` to regenerate every figure as a file.


In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import config as C
import data as D
import eda
import viz

viz.apply_style()
print("levels (hPa):", C.LEVELS_HPA)
print("station:", C.TARGET_LAT, "N", C.TARGET_LON, "E")
print("sampling:", C.STEP_HOURS, "h -> input window", C.INPUT_STEPS,
      "steps, horizons", C.HORIZON_STEPS, "steps")

## 1. Load the record

`open_raw` stitches the monthly ERA5 files into one time-sorted dataset and drops
any duplicate timestamps left behind by a retried download. `station_frame` pulls
the vertical column above the station, `add_derived` adds speed, direction and
shear, and `clean` puts the series on a gapless axis so that a missing timestamp
becomes a visible NaN rather than silently vanishing.

In [ ]:
ds = D.open_raw()
print("grid:", dict(ds.sizes))
print("period:", str(ds.valid_time.values[0])[:16], "->", str(ds.valid_time.values[-1])[:16])

raw = D.add_derived(D.station_frame(ds))
df, missing = D.clean(raw)
print("steps:", len(df), "| imputed cells:", int(missing.to_numpy().sum()))
df.head()

## 2. Statistics

The temperature column is the first sanity check: around 209 K at 50 hPa is right
for the tropical lower stratosphere, so the levels are what we asked for.

In [ ]:
stats = eda.stats_table(df)
stats

## 3. The series itself

Three things to look for: the wind is smooth rather than noisy, the zonal
component changes sign between seasons, and 10 hPa is consistently windier than
50 hPa.

In [ ]:
eda.fig_series(df)
from IPython.display import Image, display
display(Image(str(C.FIGURES / "eda_timeseries_full.png")))

## 4. Seasonal structure

The sign of the zonal wind is the regime. Over South Asia the 50 hPa flow runs
easterly through the monsoon season and reverses in winter, and that reversal is
the single largest source of variance in the record.

In [ ]:
eda.fig_monthly(df)
display(Image(str(C.FIGURES / "eda_seasonal_cycle.png")))

## 5. Wind shear between levels

Shear matters twice over. It is a predictor, because the levels above the float
altitude lead changes at it. It is also an opportunity, because an airship that
can change altitude can pick the level with the least wind.

In [ ]:
eda.fig_shear(df)
display(Image(str(C.FIGURES / "eda_wind_shear.png")))

## 6. Direction distribution

The rose shows how strongly the direction is concentrated. A concentrated rose
means a station-keeping controller mostly fights wind from a predictable
quarter.

In [ ]:
eda.fig_roses(df)
display(Image(str(C.FIGURES / "eda_wind_roses.png")))

## 7. Is this predictable at 6-24 hours?

This is the question that decides whether the project is worth doing.
Autocorrelation at the forecast horizons tells us how much of the answer
persistence already gets for free, which is exactly why persistence is the
baseline to beat rather than a straw man.

In [ ]:
eda.fig_autocorr(df)
display(Image(str(C.FIGURES / "eda_autocorrelation.png")))

## 8. Data quality

ERA5 is a reanalysis, so it is gap-free by construction; this figure exists to
prove that for this extract rather than assume it, and to catch a download that
silently dropped a month.

In [ ]:
eda.fig_missing(missing, df)
display(Image(str(C.FIGURES / "eda_missing_data.png")))

## 9. The spatial picture

The station is one column in a domain that reverses with the season. The map
gives the context a single time series cannot.

In [ ]:
eda.fig_spatial(ds)
display(Image(str(C.FIGURES / "eda_spatial_mean_flow.png")))

## 10. What this implies for the model

- The series is smooth and strongly autocorrelated at 6-24 h, so **persistence is
  a strong baseline** and any model has to be scored against it.
- The seasonal reversal dominates the variance, so the train/val/test split has
  to be **chronological**; a random split would leak the regime and flatter the
  model.
- Levels above the float altitude carry information about it, which is why the
  input is the whole **(u, v, T) column at 50/30/10 hPa** rather than 50 hPa
  alone.
- Direction is unstable when the wind is weak, so directional accuracy is scored
  only where the wind is strong enough for a bearing to mean anything.

Next: `python src/train.py`, then `python src/evaluate.py`.